# 05 · Previsões com o modelo vigente e teste integrado

**Etapa do fluxo:** 10 (previsões com o modelo vigente + evidências do ciclo).

O modelo é carregado **a partir do manifesto** `models/modelo_vigente.json`; o arquivo serializado contém o
pré-processamento ajustado e as features eleitas, então a previsão usa exatamente o mesmo tratamento do treino.
As previsões são feitas sobre o **teste reservado**, nunca usado na escolha do modelo.

In [ ]:
import sys
import warnings
from pathlib import Path

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "config.yaml").exists())
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import json
import matplotlib.pyplot as plt
import pandas as pd

from src.config import caminho, carregar_config

pd.set_option("display.max_columns", 50)
cfg = carregar_config(RAIZ)

In [ ]:
import mlflow
from sklearn.metrics import ConfusionMatrixDisplay
from src.ciclo import carregar_contexto, diretorio_ciclo, obter_cycle_id
from src.modelagem import avaliar, colunas_entrada
from src.promocao import carregar_modelo_vigente
from src.rastreamento import configurar_mlflow, run_etapa

cycle_id = obter_cycle_id(cfg)
contexto = carregar_contexto(cfg, cycle_id)
pasta_ciclo = diretorio_ciclo(cfg, cycle_id)
teste = pd.read_csv(caminho(cfg, cfg["dados"]["diretorio_processado"]) / "teste.csv")
alvo, X_cols = cfg["dados"]["alvo"], colunas_entrada(cfg)

modelo, manifesto = carregar_modelo_vigente(cfg)
assert modelo is not None, "Não há modelo vigente para gerar previsões."
colunas_do_modelo = [c for nome, _, cols in modelo.named_steps["preprocessamento"].transformers_ if nome != "remainder" for c in cols]
assert sorted(colunas_do_modelo) == sorted(manifesto["features"]), "Features do manifesto divergem do modelo serializado"
print(json.dumps({k: manifesto[k] for k in ["cycle_id", "data_version", "algoritmo", "run_id", "features", "metrica"]}, indent=2, ensure_ascii=False))

In [ ]:
probabilidade = modelo.predict_proba(teste[X_cols])[:, 1]
previsoes = pd.DataFrame({
    "registro_id": teste["registro_id"],
    "probabilidade_yes": probabilidade.round(4),
    "previsto": modelo.predict(teste[X_cols]),
    "real": teste[alvo],
})
previsoes.to_csv(pasta_ciclo / "previsoes.csv", index=False)
metricas_teste = avaliar(modelo, teste[X_cols], teste[alvo])
print("Métricas no teste reservado:", metricas_teste)
display(previsoes.sort_values("probabilidade_yes", ascending=False).head(10))
ConfusionMatrixDisplay.from_predictions(previsoes["real"], previsoes["previsto"], display_labels=["no", "yes"])
plt.title(f"Vigente {manifesto['algoritmo']} - teste")
plt.show()

## Comparação entre ciclos
Histórico persistido em `state/historico_ciclos.csv`: mostra como `data_version`, features eleitas, métricas e decisão
mudaram a cada ciclo.

In [ ]:
historico = pd.read_csv(caminho(cfg, cfg["saida"]["historico_ciclos"]))
display(historico)
if len(historico) >= 2:
    anterior, atual = historico.iloc[-2], historico.iloc[-1]
    f_ant, f_atu = set(anterior["features_eleitas"].split("|")), set(atual["features_eleitas"].split("|"))
    print("O que mudou em relação ao ciclo anterior:")
    print("  data_version:", anterior["data_version"], "->", atual["data_version"])
    print("  features que entraram:", sorted(f_atu - f_ant) or "nenhuma")
    print("  features que saíram  :", sorted(f_ant - f_atu) or "nenhuma")
    print("  decisão:", atual["decisao"], "| vigente:", atual["vigente_apos"])

## Teste integrado: o ciclo cumpriu todas as etapas?

In [ ]:
configurar_mlflow(cfg)
runs = mlflow.search_runs(filter_string=f"tags.cycle_id = '{cycle_id}'", output_format="list")
etapas = sorted(r.data.tags.get("etapa") for r in runs)
verificacoes = {
    "votação refeita neste ciclo": (pasta_ciclo / "votacao.csv").exists() and "votacao_features" in etapas,
    "dois algoritmos treinados": etapas.count("treino_candidato") == 2,
    "decisão automática registrada": manifesto["ultimo_ciclo"]["cycle_id"] == cycle_id and "decisao_promocao" in etapas,
    "mesmas features nos dois candidatos": len({r.data.params.get("features") for r in runs if r.data.tags.get("etapa") == "treino_candidato"}) == 1,
    "previsões geradas": len(previsoes) == len(teste),
}
display(pd.Series(verificacoes, name="ok").to_frame())
assert all(verificacoes.values()), "Teste integrado reprovado"

with run_etapa(cfg, contexto, "previsoes"):
    mlflow.log_params({"vigente_cycle_id": manifesto["cycle_id"], "vigente_algoritmo": manifesto["algoritmo"], "vigente_run_id": manifesto["run_id"]})
    mlflow.log_metrics({f"teste_{k}": v for k, v in metricas_teste.items() if isinstance(v, float)})
    mlflow.log_artifact(str(pasta_ciclo / "previsoes.csv"))
print("Ciclo", cycle_id, "concluído com sucesso.")